# Paso 1. Selección de filas, creación del target y depuración de columnas

**Aprendizaje de Máquina Aplicado - Trabajo Final**  
**Estudiante:** Jose Luis Bedoya Martínez  

**Objetivo del notebook:**  
Construir un dataset inicial para modelado a partir del archivo original de Resultados Únicos Saber 11, manteniendo la carga por lotes, filtrando los años 2021 y 2022, conservando únicamente estudiantes activos, creando la variable objetivo `Target` y eliminando las columnas definidas en el diccionario de datos.

**Salidas generadas:**

1. Dataset completo procesado: `DataSetInicial.csv`
2. Muestra de 7.000 registros: `DataSetInicial_muestra_7000.csv`


In [2]:
# ============================================================
# 0. Configuración inicial del entorno
# ============================================================

import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

assert sys.version_info >= (3, 7), "Este notebook requiere Python 3.7 o superior"

SEED = 42

print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro} instalado correctamente")
print(f"Pandas {pd.__version__}")


Python 3.12.13 instalado correctamente
Pandas 2.2.2


In [3]:
# ============================================================
# 0.1 Montaje de Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 1. Rutas de trabajo

Se conserva la ruta y el nombre del dataset original usado en el notebook de referencia.  
La carga se realiza por lotes para evitar problemas de memoria con el archivo completo.


In [4]:
# ============================================================
# 1. Definición de rutas
# ============================================================

INPUT_DIR = "/content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Input"
OUTPUT_DIR = "/content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# IMPORTANTE:
# Se respeta el nombre y ruta del dataset original del notebook de referencia.
ruta_csv = f"{INPUT_DIR}/Resultados_únicos_Saber_11_20260419.csv"

# Diccionario de datos usado para eliminar columnas marcadas con "Si"
ruta_diccionario = f"{INPUT_DIR}/DiccionarioDatos.xlsx"

# Archivo intermedio con filtro inicial por periodo
ruta_filtrado_periodo = f"{OUTPUT_DIR}/Resultado_filtrados_periodo_2021_2022.csv"

# Salidas finales solicitadas
ruta_salida_completa = f"{OUTPUT_DIR}/DataSetInicial.csv"
ruta_salida_muestra = f"{OUTPUT_DIR}/DataSetInicial_muestra_7000.csv"

print("Ruta dataset original:", ruta_csv)
print("Ruta diccionario:", ruta_diccionario)
print("Ruta salida completa:", ruta_salida_completa)
print("Ruta salida muestra:", ruta_salida_muestra)


Ruta dataset original: /content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Input/Resultados_únicos_Saber_11_20260419.csv
Ruta diccionario: /content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Input/DiccionarioDatos.xlsx
Ruta salida completa: /content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output/DataSetInicial.csv
Ruta salida muestra: /content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output/DataSetInicial_muestra_7000.csv


## 2. Carga por lotes y filtro por periodo

Regla aplicada:

- Se conserva únicamente el registro cuyo valor en `periodo` inicia por `2021` o `2022`.
- La carga por lotes se mantiene como en el notebook base.


In [ ]:
# ============================================================
# 2. Carga de datos por lotes y selección de registros 2021-2022
# ============================================================

anios_validos = ("2021", "2022")
chunksize = 100000

primer_bloque = True
total_leidos = 0
total_filtrados_periodo = 0

for chunk in pd.read_csv(
    ruta_csv,
    dtype=str,
    sep=",",
    low_memory=False,
    chunksize=chunksize
):
    total_leidos += len(chunk)

    # Normalizar nombres de columnas para facilitar el procesamiento posterior
    chunk.columns = chunk.columns.str.strip().str.lower()

    if "periodo" not in chunk.columns:
        raise ValueError(
            "No se encontró la columna 'periodo'. "
            f"Columnas disponibles: {list(chunk.columns)}"
        )

    # Se eliminan registros cuyo periodo no inicie por 2021 o 2022
    chunk["periodo"] = chunk["periodo"].astype(str).str.strip()
    chunk_filtrado = chunk[
        chunk["periodo"].str[:4].isin(anios_validos)
    ].copy()

    total_filtrados_periodo += len(chunk_filtrado)

    # Guardar acumulando los bloques filtrados
    chunk_filtrado.to_csv(
        ruta_filtrado_periodo,
        mode="w" if primer_bloque else "a",
        header=primer_bloque,
        index=False,
        encoding="utf-8-sig"
    )

    primer_bloque = False

print("Proceso de filtro por periodo finalizado.")
print(f"Total de registros leídos: {total_leidos:,}")
print(f"Total de registros filtrados 2021-2022: {total_filtrados_periodo:,}")
print("Archivo intermedio guardado en:", ruta_filtrado_periodo)


Proceso de filtro por periodo finalizado.
Total de registros leídos: 7,109,704
Total de registros filtrados 2021-2022: 1,101,465
Archivo intermedio guardado en: /content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output/Resultado_filtrados_periodo_2021_2022.csv


## 3. Carga del archivo filtrado

Después del filtro inicial por periodo, se carga el archivo intermedio para aplicar las reglas restantes de depuración y construcción del target.


In [5]:
# ============================================================
# 3. Cargar archivo intermedio filtrado por periodo
# ============================================================

df = pd.read_csv(
    ruta_filtrado_periodo,
    dtype=str,
    sep=",",
    low_memory=False
)

df.columns = df.columns.str.strip().str.lower()

print("Dimensión inicial después de filtro por periodo:", df.shape)
display(df.head())


Dimensión inicial después de filtro por periodo: (1101465, 51)


,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20224,TI,SB11202240536228,URBANO,N,A,TÉCNICO/ACADÉMICO,183247000272,183247000272,18,...,No,No,Si,A2,60,44,50,41,50,237
1,20224,TI,SB11202240549969,URBANO,N,A,TÉCNICO/ACADÉMICO,163001002496,163001002496,63,...,No,Si,Si,A-,44,42,46,43,40,214
2,20211,TI,SB11202110001547,RURAL,S,B,ACADÉMICO,425758800009,425758800009,25,...,Si,Si,Si,B+,83,74,64,61,69,341
3,20211,CR,SB11202110021004,URBANO,NaN,A,ACADÉMICO,425799000637,425799000637,25,...,Si,Si,Si,A2,66,59,60,56,66,303
4,20224,TI,SB11202240101947,RURAL,NaN,A,NO APLICA,205034000329,205034001457,05,...,No,No,No,A1,50,58,52,55,59,278


## 4. Funciones de limpieza, target y aplicación del diccionario

In [6]:
import numpy as np
# ============================================================
# 4. Funciones auxiliares
# ============================================================

def validar_columnas_requeridas(df, columnas):
    """Valida que existan las columnas requeridas en el DataFrame."""
    faltantes = [col for col in columnas if col not in df.columns]
    if faltantes:
        raise ValueError(
            f"No se encontraron estas columnas requeridas: {faltantes}. "
            f"Columnas disponibles: {list(df.columns)}"
        )


def filtrar_estudiantes_activos(df, columna="estu_estudiante"):
    """
    Elimina registros donde estu_estudiante sea 'N'.
    La regla conserva los registros diferentes de 'N', incluyendo 'S'.
    """
    validar_columnas_requeridas(df, [columna])

    df = df.copy()
    total_antes = len(df)

    valor_normalizado = df[columna].astype(str).str.strip().str.upper()
    df = df[valor_normalizado != "N"].copy()

    total_despues = len(df)
    print(f"Registros antes de filtrar {columna}='N': {total_antes:,}")
    print(f"Registros eliminados: {total_antes - total_despues:,}")
    print(f"Registros finales: {total_despues:,}")

    return df


def crear_puntaje_y_target(df):
    """
    Crea la variable Puntaje a partir de los puntajes individuales Saber 11
    y luego crea la variable Target con cuatro niveles de desempeño.

    Fórmula:
    Puntaje = (
        punt_ingles * 1
        + punt_matematicas * 3
        + punt_sociales_ciudadanas * 3
        + punt_c_naturales * 3
        + punt_lectura_critica * 3
    ) / 13
    """

    columnas_puntajes = [
        "punt_ingles",
        "punt_matematicas",
        "punt_sociales_ciudadanas",
        "punt_c_naturales",
        "punt_lectura_critica",
    ]

    validar_columnas_requeridas(df, columnas_puntajes)

    df = df.copy()

    # Conversión robusta a numérico
    for col in columnas_puntajes:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    total_antes = len(df)

    # Se eliminan registros con puntajes nulos, no convertibles o negativos
    df = df.dropna(subset=columnas_puntajes).copy()

    for col in columnas_puntajes:
        df = df[df[col] >= 0].copy()

    print(
        "Registros eliminados por puntajes nulos/no convertibles/negativos: "
        f"{total_antes - len(df):,}"
    )

    # Cálculo ponderado del puntaje
    df["Puntaje"] = (
        df["punt_ingles"] * 1
        + df["punt_matematicas"] * 3
        + df["punt_sociales_ciudadanas"] * 3
        + df["punt_c_naturales"] * 3
        + df["punt_lectura_critica"] * 3
    ) / 13

    # Clasificación en niveles
    condiciones = [
        (df["Puntaje"] >= 0) & (df["Puntaje"] <= 40),
        (df["Puntaje"] > 40) & (df["Puntaje"] <= 55),
        (df["Puntaje"] > 55) & (df["Puntaje"] <= 70),
        (df["Puntaje"] > 70) & (df["Puntaje"] <= 100),
    ]

    clases = [
        "Nivel 1: Insuficiente",
        "Nivel 2: Mínimo",
        "Nivel 3: Satisfactorio",
        "Nivel 4: Avanzado",
    ]

    df["Target"] = np.select(condiciones, clases, default='Sin clasificar')

    total_antes_target = len(df)
    df = df[df["Target"].notna()].copy()

    print(
        "Registros eliminados por quedar fuera de los rangos del Target: "
        f"{total_antes_target - len(df):,}"
    )

    print("Distribución de Target:")
    display(df["Target"].value_counts(dropna=False).to_frame("Registros"))

    # Eliminación de variables solicitadas
    columnas_a_eliminar = ["Puntaje", "punt_global"]
    df = df.drop(columns=[c for c in columnas_a_eliminar if c in df.columns])

    return df

def aplicar_diccionario_eliminacion(df, ruta_diccionario):
    """
    Elimina del DataFrame las columnas que en el archivo DiccionarioDatos.xlsx
    tengan el valor 'Si' en la columna 'Eliminar'.

    Nota metodológica:
    - El diccionario documenta las variables originales.
    - Las variables derivadas no presentes en el diccionario, como Target, se conservan.
    """
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()

    dic = pd.read_excel(ruta_diccionario)
    dic.columns = dic.columns.str.strip()

    columnas_esperadas = ["Nombre de la columna", "Eliminar"]
    faltantes = [c for c in columnas_esperadas if c not in dic.columns]
    if faltantes:
        raise ValueError(f"El diccionario no contiene estas columnas requeridas: {faltantes}")

    dic["Nombre de la columna"] = (
        dic["Nombre de la columna"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dic["Eliminar"] = (
        dic["Eliminar"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    columnas_eliminar_dic = dic.loc[
        dic["Eliminar"] == "si",
        "Nombre de la columna"
    ].tolist()

    columnas_eliminar_presentes = [
        col for col in columnas_eliminar_dic
        if col in df.columns
    ]

    df = df.drop(columns=columnas_eliminar_presentes, errors="ignore")

    print("Columnas marcadas para eliminar en el diccionario:", len(columnas_eliminar_dic))
    print("Columnas eliminadas presentes en el dataset:", len(columnas_eliminar_presentes))
    print(columnas_eliminar_presentes)

    return df


def eliminar_variables_intermedias(df):
    """
    Elimina las variables solicitadas:
    - Puntaje
    - punt_global

    Se usa errors='ignore' para que el notebook no falle si alguna ya fue eliminada
    por el diccionario.
    """
    df = df.copy()
    columnas_a_eliminar = ["Puntaje", "puntaje", "punt_global", "puntaje global", "puntaje_global"]
    df = df.drop(columns=columnas_a_eliminar, errors="ignore")
    return df


def obtener_muestra_7000(df, target_col="Target", n=7000, random_state=SEED):
    """
    Obtiene una muestra reproducible de 7.000 registros.

    Si existe la variable Target y cada clase tiene registros suficientes,
    intenta tomar una muestra estratificada para conservar la distribución
    de las clases. Si no es posible, toma una muestra aleatoria simple.
    """
    if len(df) <= n:
        print(f"El dataset tiene {len(df):,} registros. Se devuelve completo.")
        return df.copy()

    if target_col in df.columns:
        try:
            muestra = (
                df.groupby(target_col, group_keys=False)
                  .apply(lambda x: x.sample(
                      n=max(1, round(n * len(x) / len(df))),
                      random_state=random_state
                  ))
                  .sample(frac=1, random_state=random_state)
                  .reset_index(drop=True)
            )

            # Ajuste por redondeo para dejar exactamente n registros
            if len(muestra) > n:
                muestra = muestra.sample(n=n, random_state=random_state)
            elif len(muestra) < n:
                faltantes = n - len(muestra)
                complemento = df.drop(index=muestra.index, errors="ignore").sample(
                    n=faltantes,
                    random_state=random_state
                )
                muestra = pd.concat([muestra, complemento], ignore_index=True)

            print("Muestra estratificada generada.")
            print("Distribución de Target en la muestra:")
            display(muestra[target_col].value_counts(dropna=False).to_frame("Registros"))
            return muestra.reset_index(drop=True)

        except Exception as e:
            print("No fue posible generar muestra estratificada. Se usará muestra aleatoria simple.")
            print("Detalle:", e)

    return df.sample(n=n, random_state=random_state).reset_index(drop=True)

## 5. Aplicación de reglas de negocio y creación del target

In [7]:
# ============================================================
# 5. Aplicar reglas solicitadas
# ============================================================

print("Dimensión inicial:", df.shape)

# b. El periodo ya fue filtrado por lotes; se valida nuevamente por seguridad
df["periodo"] = df["periodo"].astype(str).str.strip()
df = df[df["periodo"].str[:4].isin(("2021", "2022"))].copy()
print("Dimensión después de validar periodo 2021-2022:", df.shape)

# c. Eliminar registros con estu_estudiante = 'N'
df = filtrar_estudiantes_activos(df, columna="estu_estudiante")
print("Dimensión después de filtrar estudiantes activos:", df.shape)

# d-e. Convertir punt_global a numérico, crear Puntaje y Target
df = crear_puntaje_y_target(df)
print("Dimensión después de crear Puntaje y Target:", df.shape)


Dimensión inicial: (1101465, 51)
Dimensión después de validar periodo 2021-2022: (1101465, 51)
Registros antes de filtrar estu_estudiante='N': 1,101,465
Registros eliminados: 0
Registros finales: 1,101,465
Dimensión después de filtrar estudiantes activos: (1101465, 51)
Registros eliminados por puntajes nulos/no convertibles/negativos: 4,105
Registros eliminados por quedar fuera de los rangos del Target: 0
Distribución de Target:


,Registros
Target,
Nivel 2: Mínimo,527626
Nivel 3: Satisfactorio,325112
Nivel 1: Insuficiente,205935
Nivel 4: Avanzado,38687


Dimensión después de crear Puntaje y Target: (1097360, 51)


## 6. Eliminación de columnas según el diccionario

Primero se eliminan las columnas marcadas con `Si` en el archivo `DiccionarioDatos.xlsx`.  
Luego se eliminan explícitamente las variables `Puntaje` y `punt_global`, para cumplir con la regla solicitada y evitar fuga de información hacia el modelo.


In [8]:
# ============================================================
# 6. Aplicar diccionario y eliminar variables intermedias
# ============================================================

df_modelo = aplicar_diccionario_eliminacion(
    df=df,
    ruta_diccionario=ruta_diccionario
)

# f. Eliminar Puntaje y punt_global
df_modelo = eliminar_variables_intermedias(df_modelo)

print("Dimensión final del dataset:", df_modelo.shape)
print("Columnas finales:")
print(df_modelo.columns.tolist())

display(df_modelo.head())


Columnas marcadas para eliminar en el diccionario: 19
Columnas eliminadas presentes en el dataset: 18
['cole_cod_dane_establecimiento', 'cole_cod_dane_sede', 'cole_cod_depto_ubicacion', 'cole_cod_mcpio_ubicacion', 'cole_codigo_icfes', 'cole_nombre_establecimiento', 'cole_nombre_sede', 'estu_cod_depto_presentacion', 'estu_cod_mcpio_presentacion', 'estu_cod_reside_depto', 'estu_cod_reside_mcpio', 'estu_fechanacimiento', 'desemp_ingles', 'punt_ingles', 'punt_matematicas', 'punt_sociales_ciudadanas', 'punt_c_naturales', 'punt_lectura_critica']
Dimensión final del dataset: (1097360, 33)
Columnas finales:
['periodo', 'estu_tipodocumento', 'estu_consecutivo', 'cole_area_ubicacion', 'cole_bilingue', 'cole_calendario', 'cole_caracter', 'cole_depto_ubicacion', 'cole_genero', 'cole_jornada', 'cole_mcpio_ubicacion', 'cole_naturaleza', 'cole_sede_principal', 'estu_depto_presentacion', 'estu_depto_reside', 'estu_estadoinvestigacion', 'estu_estudiante', 'estu_genero', 'estu_mcpio_presentacion', 'estu

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_depto_ubicacion,cole_genero,cole_jornada,...,fami_cuartoshogar,fami_educacionmadre,fami_educacionpadre,fami_estratovivienda,fami_personashogar,fami_tieneautomovil,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,target
0,20224,TI,SB11202240536228,URBANO,N,A,TÉCNICO/ACADÉMICO,CAQUETA,MIXTO,UNICA,...,Cuatro,Primaria incompleta,Primaria incompleta,Estrato 2,5 a 6,No,No,No,Si,Nivel 2: Mínimo
1,20224,TI,SB11202240549969,URBANO,N,A,TÉCNICO/ACADÉMICO,QUINDIO,MIXTO,MAÑANA,...,Dos,Educación profesional completa,No Aplica,Estrato 2,3 a 4,No,No,Si,Si,Nivel 2: Mínimo
2,20211,TI,SB11202110001547,RURAL,S,B,ACADÉMICO,CUNDINAMARCA,MIXTO,COMPLETA,...,Tres,Educación profesional completa,Postgrado,Estrato 4,3 a 4,Si,Si,Si,Si,Nivel 3: Satisfactorio
3,20211,CR,SB11202110021004,URBANO,NaN,A,ACADÉMICO,CUNDINAMARCA,MIXTO,MAÑANA,...,Tres,Educación profesional completa,Educación profesional completa,Estrato 5,1 a 2,Si,Si,Si,Si,Nivel 3: Satisfactorio
4,20224,TI,SB11202240101947,RURAL,NaN,A,NO APLICA,ANTIOQUIA,MIXTO,COMPLETA,...,Dos,Primaria incompleta,Primaria incompleta,Estrato 2,3 a 4,No,No,No,No,Nivel 3: Satisfactorio


In [ ]:
columnas_finales = pd.DataFrame({'Columnas': df_modelo.columns})
display(columnas_finales)

,Columnas
0,periodo
1,estu_tipodocumento
2,estu_consecutivo
3,cole_area_ubicacion
4,cole_bilingue
5,cole_calendario
6,cole_caracter
7,cole_depto_ubicacion
8,cole_genero
9,cole_jornada


## 7. Una vez tenemos las columnas elegidas, vamos a remover registros duplicados

In [1]:
import pandas as pd

def eliminar_registros_repetidos(
    df: pd.DataFrame,
    columna_id: str = "estu_consecutivo",
    keep: str = "first",
    mostrar_resumen: bool = True
):
    """
    Identifica y elimina registros repetidos en un DataFrame usando una columna identificadora.

    En este caso, un registro se considera repetido cuando existen dos o más filas
    con el mismo valor en la columna 'estu_consecutivo', ya que esta representa
    el número de identificación del estudiante.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset original.

    columna_id : str
        Columna usada para identificar duplicados. Por defecto: 'estu_consecutivo'.

    keep : str
        Define qué registro conservar cuando hay duplicados.
        Opciones:
        - 'first': conserva el primer registro encontrado.
        - 'last': conserva el último registro encontrado.
        - False: elimina todos los registros que estén duplicados.

    mostrar_resumen : bool
        Si es True, imprime un resumen del proceso.

    Retorna
    -------
    df_limpio : pd.DataFrame
        Dataset sin registros repetidos.

    df_repetidos : pd.DataFrame
        Registros identificados como repetidos.
    """

    # Validar que la columna exista en el dataset
    if columna_id not in df.columns:
        raise ValueError(f"La columna '{columna_id}' no existe en el DataFrame.")

    # Copia para evitar modificar el DataFrame original
    df_temp = df.copy()

    # Identificar todos los registros que tienen un identificador repetido
    df_repetidos = df_temp[
        df_temp.duplicated(subset=[columna_id], keep=False)
    ].sort_values(by=columna_id)

    # Eliminar duplicados conservando el criterio definido en keep
    df_limpio = df_temp.drop_duplicates(
        subset=[columna_id],
        keep=keep
    ).reset_index(drop=True)

    if mostrar_resumen:
        total_registros = len(df_temp)
        total_repetidos = len(df_repetidos)
        total_ids_repetidos = df_repetidos[columna_id].nunique()
        total_eliminados = total_registros - len(df_limpio)

        print("Resumen de depuración de registros repetidos")
        print("-" * 50)
        print(f"Total de registros originales: {total_registros:,}")
        print(f"Total de registros después de depurar: {len(df_limpio):,}")
        print(f"Total de registros eliminados: {total_eliminados:,}")
        print(f"Total de filas involucradas en duplicados: {total_repetidos:,}")
        print(f"Total de estudiantes con registros repetidos: {total_ids_repetidos:,}")

    return df_limpio, df_repetidos

In [9]:
df_limpio, df_repetidos = eliminar_registros_repetidos(
    df_modelo,
    columna_id="estu_consecutivo",
    keep="first"
)

Resumen de depuración de registros repetidos
--------------------------------------------------
Total de registros originales: 1,097,360
Total de registros después de depurar: 566,241
Total de registros eliminados: 531,119
Total de filas involucradas en duplicados: 1,061,934
Total de estudiantes con registros repetidos: 530,815


In [10]:
df_repetidos.head(20)

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_depto_ubicacion,cole_genero,cole_jornada,...,fami_cuartoshogar,fami_educacionmadre,fami_educacionpadre,fami_estratovivienda,fami_personashogar,fami_tieneautomovil,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,target
438681,20224,CC,SB11202240000034,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Dos,Primaria incompleta,Primaria completa,Sin Estrato,3 a 4,No,Si,No,Si,Nivel 1: Insuficiente
968317,20224,CC,SB11202240000034,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Dos,Primaria incompleta,Primaria completa,Sin Estrato,3 a 4,No,Si,No,Si,Nivel 1: Insuficiente
723098,20224,CC,SB11202240000036,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Uno,Primaria incompleta,Primaria incompleta,Estrato 3,3 a 4,No,Si,Si,No,Nivel 1: Insuficiente
193475,20224,CC,SB11202240000036,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Uno,Primaria incompleta,Primaria incompleta,Estrato 3,3 a 4,No,Si,Si,No,Nivel 1: Insuficiente
735365,20224,CC,SB11202240000037,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Seis o mas,Primaria completa,Primaria completa,Estrato 5,9 o más,No,NaN,No,Si,Nivel 1: Insuficiente
205749,20224,CC,SB11202240000037,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Seis o mas,Primaria completa,Primaria completa,Estrato 5,9 o más,No,NaN,No,Si,Nivel 1: Insuficiente
218308,20224,CC,SB11202240000039,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Dos,Secundaria (Bachillerato) incompleta,Secundaria (Bachillerato) incompleta,Estrato 2,5 a 6,No,No,No,No,Nivel 1: Insuficiente
747924,20224,CC,SB11202240000039,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Dos,Secundaria (Bachillerato) incompleta,Secundaria (Bachillerato) incompleta,Estrato 2,5 a 6,No,No,No,No,Nivel 1: Insuficiente
989523,20224,CC,SB11202240000040,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Cuatro,NaN,NaN,NaN,3 a 4,No,No,NaN,Si,Nivel 1: Insuficiente
459876,20224,CC,SB11202240000040,URBANO,N,A,ACADÉMICO,CHOCO,MIXTO,MAÑANA,...,Cuatro,NaN,NaN,NaN,3 a 4,No,No,NaN,Si,Nivel 1: Insuficiente


In [11]:
duplicados_restantes = df_limpio["estu_consecutivo"].duplicated().sum()

print(f"Duplicados restantes en estu_consecutivo: {duplicados_restantes}")

Duplicados restantes en estu_consecutivo: 0


In [12]:
df_modelo = df_limpio.copy()

In [13]:
print("Distribución de la variable 'Target':")
display(df_modelo['target'].value_counts().to_frame('Cantidad'))

Distribución de la variable 'Target':


,Cantidad
target,
Nivel 2: Mínimo,268741
Nivel 3: Satisfactorio,170346
Nivel 1: Insuficiente,104523
Nivel 4: Avanzado,22631


## 7. Guardar salidas solicitadas

In [14]:
# ============================================================
# 7. Guardar CSV con todos los datos obtenidos
# ============================================================

df_modelo.to_csv(
    ruta_salida_completa,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset completo guardado en:")
print(ruta_salida_completa)
print("Dimensión guardada:", df_modelo.shape)


Dataset completo guardado en:
/content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output/DataSetInicial.csv
Dimensión guardada: (566241, 33)


In [15]:
# ============================================================
# 8. Crear y guardar CSV con muestra de 7.000 registros
# ============================================================

df_muestra_7000 = obtener_muestra_7000(
    df=df_modelo,
    target_col="Target",
    n=7000,
    random_state=SEED
)

df_muestra_7000.to_csv(
    ruta_salida_muestra,
    index=False,
    encoding="utf-8-sig"
)

print("Muestra de 7.000 registros guardada en:")
print(ruta_salida_muestra)
print("Dimensión guardada:", df_muestra_7000.shape)


Muestra de 7.000 registros guardada en:
/content/drive/MyDrive/Aprendizaje de Maquina Aplicado/TrabajoFinal/Data/Output/DataSetInicial_muestra_7000.csv
Dimensión guardada: (7000, 33)


## 8. Validaciones finales

Estas validaciones permiten confirmar que:

- Solo quedan periodos 2021 y 2022.
- No quedan registros con `estu_estudiante = N`, si la columna se conserva.
- La variable `Target` fue creada.
- Las variables `Puntaje` y `punt_global` no quedan en el dataset final.


In [16]:
# ============================================================
# 9. Validaciones finales
# ============================================================

print("Validación de años en periodo:")
display(df_modelo["periodo"].astype(str).str[:4].value_counts().to_frame("Registros"))

if "estu_estudiante" in df_modelo.columns:
    print("\nValidación de estu_estudiante:")
    display(df_modelo["estu_estudiante"].astype(str).str.upper().value_counts(dropna=False).to_frame("Registros"))

print("\nDistribución final del Target:")
print(df_modelo["target"].value_counts(dropna=False))

columnas_prohibidas = [c for c in ["Puntaje", "puntaje", "punt_global", "puntaje_global"] if c in df_modelo.columns]
print("\nColumnas que no deberían quedar en el dataset:", columnas_prohibidas)

assert df_modelo["periodo"].astype(str).str[:4].isin(["2021", "2022"]).all()
assert "target" in df_modelo.columns # Corregido de 'Target' a 'target'
assert len(columnas_prohibidas) == 0

print("\nValidaciones completadas correctamente.")

Validación de años en periodo:


,Registros
periodo,
2022,550760
2021,15481



Validación de estu_estudiante:


,Registros
estu_estudiante,
ESTUDIANTE,566241



Distribución final del Target:
target
Nivel 2: Mínimo           268741
Nivel 3: Satisfactorio    170346
Nivel 1: Insuficiente     104523
Nivel 4: Avanzado          22631
Name: count, dtype: int64

Columnas que no deberían quedar en el dataset: []

Validaciones completadas correctamente.
